# 3강: PyQt6 GUI 기초 실습

이 노트북에서는 PyQt6의 핵심 개념을 배우고,
SeekSeek에서 사용하는 Model/View 패턴, 시그널/슬롯, QThread를 실습합니다.

> **참고**: PyQt6 GUI 코드는 이벤트 루프가 필요하므로 일부 셀은
> 별도 `.py` 파일로 실행해야 동작합니다.

## 3.1 시그널/슬롯 메커니즘

Qt의 시그널/슬롯은 객체 간 통신 패턴입니다:
- **시그널(Signal)**: 이벤트 발생을 알리는 함수 (emit)
- **슬롯(Slot)**: 시그널에 연결된 핸들러 함수
- **connect()**: 시그널과 슬롯을 연결

```python
# 시그널 정의
class Worker(QThread):
    progress = pyqtSignal(str, int)  # (메시지, 진행률)
    finished = pyqtSignal(int)       # 완료 건수

# 슬롯 연결
worker.progress.connect(self.on_progress)   # 크로스 스레드 안전!
worker.finished.connect(self.on_finished)
```

크로스 스레드 시그널은 Qt가 자동으로 이벤트 큐를 통해 마샬링합니다.

## 3.2 QAbstractTableModel — 가상 렌더링

SeekSeek의 `ResultTableModel`은 QAbstractTableModel을 상속합니다.

### 가상 렌더링의 장점
- 100만 건 결과도 O(1)으로 표시 (화면에 보이는 행만 data() 호출)
- QStandardItemModel 대비 메모리 사용량 1/10 이하
- 데이터 변환(포맷팅, 아이콘)을 data()에서 on-demand 처리

In [ ]:
# ResultTableModel 구현 분석 (실행 가능한 미니 버전)

from dataclasses import dataclass

@dataclass
class SearchResult:
    """검색 결과 하나를 나타내는 데이터 클래스"""
    file_id: int
    path: str
    name: str
    extension: str
    size: int
    modified: float
    match_type: str    # 'filename', 'content', 'both'
    is_dir: bool = False

# 테스트 데이터
results = [
    SearchResult(1, r'C:\test\main.py', 'main.py', '.py', 1024, 1700000000.0, 'filename'),
    SearchResult(2, r'C:\docs\report.docx', 'report.docx', '.docx', 52000, 1699000000.0, 'content'),
    SearchResult(3, r'C:\test\config.py', 'config.py', '.py', 512, 1698000000.0, 'both'),
]

# Model/View 패턴의 핵심: data() 함수
COL_LABELS = ['이름', '경로', '크기', '수정일', '확장자', '매칭']

def format_size(size: int) -> str:
    """바이트 → 사람이 읽기 쉬운 단위로 변환"""
    for unit in ('B', 'KB', 'MB', 'GB'):
        if size < 1024:
            return f'{size:.0f} {unit}' if unit == 'B' else f'{size:.1f} {unit}'
        size /= 1024
    return f'{size:.1f} TB'

def data(row: int, col: int) -> str:
    """QAbstractTableModel.data() 시뮬레이션"""
    r = results[row]
    import os
    from datetime import datetime
    if col == 0: return r.name
    if col == 1: return os.path.dirname(r.path)
    if col == 2: return format_size(r.size)
    if col == 3: return datetime.fromtimestamp(r.modified).strftime('%Y-%m-%d %H:%M')
    if col == 4: return r.extension
    if col == 5: return {'filename': '파일명', 'content': '본문', 'both': '파일명+본문'}[r.match_type]
    return ''

# 테이블 출력 시뮬레이션
print(' | '.join(f'{h:12s}' for h in COL_LABELS))
print('-' * 80)
for row in range(len(results)):
    print(' | '.join(f'{data(row, col):12s}' for col in range(6)))

## 3.3 QThread 패턴 — 백그라운드 작업

SeekSeek의 스레드 구조:

```
┌─ GUI 스레드 (MainWindow) ─────────────────────────┐
│  사용자 입력 → 검색 → 결과 표시                      │
│  시그널 수신 → UI 업데이트                           │
├──────────────────────────────────────────────────────┤
│  ↓ pyqtSignal (자동 마샬링)                         │
├──────────────────────────────────────────────────────┤
│  ScannerThread      → MFT 열거 + 캐시 구축          │
│  USNMonitorThread   → 5초 폴링 → 변경 캐시 갱신      │
│  ContentReindexThread → 병렬 텍스트 추출 + DB 저장    │
│  FolderIndexThread  → 폴더 탐색 + 색인               │
└──────────────────────────────────────────────────────┘
```

### v1.0.1 이후 동작 메모

- ContentReindexThread 워커 수는 CPU 기준 `2~4`로 제한됩니다.
- 추출 결과는 청크 전체를 모으지 않고 소배치로 즉시 DB에 flush합니다.
- 전체 색인과 변경분 색인은 동시 실행이 아니라 직렬 실행됩니다.

In [ ]:
# QThread 시뮬레이션 (콘솔 환경에서 실행 가능하도록 threading 사용)
import threading
import time

class MockScannerThread:
    """ScannerThread의 동작을 시뮬레이션하는 클래스"""
    
    def __init__(self):
        self._stop_requested = False
        self._callbacks = {'progress': [], 'finished': []}
    
    def connect_progress(self, callback):
        self._callbacks['progress'].append(callback)
    
    def connect_finished(self, callback):
        self._callbacks['finished'].append(callback)
    
    def request_stop(self):
        self._stop_requested = True
    
    def run(self):
        """MFT 스캔 시뮬레이션"""
        total = 100000
        for i in range(0, total, 25000):
            if self._stop_requested:
                break
            time.sleep(0.1)  # 실제로는 MFT 읽기
            for cb in self._callbacks['progress']:
                cb(f'스캔 중... {i:,}/{total:,}', i)
        for cb in self._callbacks['finished']:
            cb(total, 0)

# 사용
scanner = MockScannerThread()
scanner.connect_progress(lambda msg, n: print(f'  진행: {msg}'))
scanner.connect_finished(lambda total, _: print(f'  완료: {total:,}개 파일'))

thread = threading.Thread(target=scanner.run)
thread.start()
thread.join()
print('스캔 시뮬레이션 완료')

## 3.4 검색 결과 정렬 — 우선순위 시스템

SeekSeek은 검색 결과를 다중 기준으로 정렬합니다:
1. `match_type` 우선순위: both(0) > filename(1) > content(2)
2. 디렉터리 우선 (is_dir)
3. 파일명 알파벳순

In [ ]:
# 정렬 로직 실습
priority = {'both': 0, 'filename': 1, 'content': 2}

test_results = [
    SearchResult(1, 'a.py',   'a.py',   '.py', 100, 0, 'content'),
    SearchResult(2, 'b.py',   'b.py',   '.py', 200, 0, 'both'),
    SearchResult(3, 'c/',     'c',      '',    0,   0, 'filename', is_dir=True),
    SearchResult(4, 'd.txt',  'd.txt',  '.txt', 50, 0, 'filename'),
    SearchResult(5, 'e.py',   'e.py',   '.py', 150, 0, 'both'),
]

sorted_results = sorted(
    test_results,
    key=lambda r: (priority.get(r.match_type, 9), not r.is_dir, r.name.lower()),
)

print('정렬 전:')
for r in test_results:
    print(f'  {r.name:10s} match={r.match_type:10s} dir={r.is_dir}')

print('\n정렬 후:')
for r in sorted_results:
    print(f'  {r.name:10s} match={r.match_type:10s} dir={r.is_dir}')

## 3.5 미리보기 — 키워드 하이라이팅

SeekSeek은 검색 키워드를 HTML `<span>` 태그로 감싸 배경색을 입힙니다.

In [ ]:
import html
import re

def highlight_text(text: str, keywords: list[str]) -> str:
    """텍스트에서 키워드를 HTML 하이라이트로 감싸기"""
    escaped = html.escape(text)
    for kw in keywords:
        if not kw:
            continue
        pattern = re.escape(html.escape(kw))
        escaped = re.sub(
            pattern,
            lambda m: f'<span style="background:#ffeb3b;font-weight:bold">{m.group()}</span>',
            escaped,
            flags=re.IGNORECASE,
        )
    return f'<pre>{escaped}</pre>'

sample_text = """import os
from core.searcher import search

results = search(filename_query='main.py', content_query='import')
for r in results:
    print(r.path)
"""

highlighted = highlight_text(sample_text, ['import', 'search'])
print('원본:')
print(sample_text)
print('하이라이트된 HTML:')
print(highlighted)

## 요약

- **시그널/슬롯**: Qt의 이벤트 통신 패턴, 크로스 스레드 자동 마샬링
- **QAbstractTableModel**: 가상 렌더링으로 대량 데이터 효율적 표시
- **QThread**: 백그라운드 작업을 GUI와 분리, pyqtSignal로 결과 전달
- **정렬**: match_type 우선순위 → 디렉터리 → 파일명 알파벳순
- **키워드 하이라이팅**: HTML span + CSS background로 시각적 강조